In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

from google.colab import files
from PIL import Image

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cpu


In [ ]:
train_transforms = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(20),

    transforms.RandomAffine(
        degrees=15,
        scale=(0.9,1.1)
    ),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

val_test_transforms = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor(),

    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

In [ ]:
train_data = datasets.ImageFolder(
    "/content/dataset_split/train",
    transform=train_transforms
)

val_data = datasets.ImageFolder(
    "/content/dataset_split/val",
    transform=val_test_transforms
)

test_data = datasets.ImageFolder(
    "/content/dataset_split/test",
    transform=val_test_transforms
)

train_loader = DataLoader(
    train_data,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_data,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_data,
    batch_size=32,
    shuffle=False
)

class_names = train_data.classes

print(class_names)

FileNotFoundError: [Errno 2] No such file or directory: '/content/dataset_split/train'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls "/content/drive/MyDrive"

'2023 - 2024 Backtest .xlsx'
'2025 12 05 kavindi'
'2025 12 05 matara'
 93d8ea24022cfbe.txt
 af-2-kalhara-ediriweera.rar
'BOC (1).pdf'
 BOC.pdf
 Classroom
'Colab Notebooks'
'cricket04 (1).zip'
 cricket04.zip
 dataset
'Draw diagram s showing equivalence partitions and boundary values that ensure all input'$'\n''values are tested forthe system. Clearly mention the values.'$'\n''• ForAge,'$'\n\n''• ForAmount:.gsheet'
'Family Update (1).gsite'
'Family Update.gsite'
 IMG-20210521-WA0024.jpeg
'IMG-20230227-WA0021 (1).jpg'
 IMG-20230227-WA0021.jpg
 IMG_5150.JPG
'kavindi photos 11 03'
 Malabe
 NEWW.drawio.png
 pest_model.h5
'Presentation (10).pptx'
'research pdf'
 ResultSheet.pdf
'Scan 10 Jul 23 · 07·53·42.pdf'
 suddi
'UEE VIDEO Recording'
'Untitled document.gdoc'
'Untitled site.gsite'
'WhatsApp Image 2023-07-10 at 19.49.15.jpg'


In [ ]:
train_data = datasets.ImageFolder(
    "/content/drive/MyDrive/dataset/train",
    transform=train_transforms
)

val_data = datasets.ImageFolder(
    "/content/drive/MyDrive/dataset/val",
    transform=val_test_transforms
)

test_data = datasets.ImageFolder(
    "/content/drive/MyDrive/dataset/test",
    transform=val_test_transforms
)

train_loader = DataLoader(
    train_data,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_data,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_data,
    batch_size=32,
    shuffle=False
)

class_names = train_data.classes

print("✅ Classes:", class_names)

print("✅ Train Images:", len(train_data))
print("✅ Validation Images:", len(val_data))
print("✅ Test Images:", len(test_data))

✅ Classes: ['Brown Planthopper', 'Rice Gall Midge', 'Rice Hispa', 'Rice Leaf Folder', 'Rice Stem Borer']
✅ Train Images: 953
✅ Validation Images: 204
✅ Test Images: 208


In [ ]:
model = models.mobilenet_v2(weights="DEFAULT")

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 63.3MB/s]


In [ ]:
for param in model.parameters():
    param.requires_grad = False

In [ ]:
for param in model.features[14:].parameters():
    param.requires_grad = True

In [ ]:
num_features = model.classifier[1].in_features

model.classifier = nn.Sequential(

    nn.Dropout(0.4),

    nn.Linear(num_features, 5)
)

model = model.to(device)

print(model)

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)

scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=5,
    gamma=0.5
)

In [ ]:
best_acc = 0

for epoch in range(20):

    model.train()

    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    # VALIDATION

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, preds = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (preds == labels).sum().item()

    val_acc = 100 * correct / total

    print(f"🔥 Epoch {epoch+1} | Loss: {running_loss:.2f} | Val Acc: {val_acc:.2f}%")

    # SAVE BEST MODEL

    if val_acc > best_acc:

        best_acc = val_acc

        torch.save(
            model.state_dict(),
            "mobilenetv2_best.pth"
        )

        print(f"✅ Best Model Saved: {best_acc:.2f}%")

    scheduler.step()

🔥 Epoch 1 | Loss: 44.63 | Val Acc: 57.35%
✅ Best Model Saved: 57.35%
🔥 Epoch 2 | Loss: 33.37 | Val Acc: 62.25%
✅ Best Model Saved: 62.25%
🔥 Epoch 3 | Loss: 22.29 | Val Acc: 64.22%
✅ Best Model Saved: 64.22%
🔥 Epoch 4 | Loss: 14.45 | Val Acc: 69.61%
✅ Best Model Saved: 69.61%
🔥 Epoch 5 | Loss: 10.58 | Val Acc: 73.53%
✅ Best Model Saved: 73.53%
🔥 Epoch 6 | Loss: 8.58 | Val Acc: 73.04%
🔥 Epoch 7 | Loss: 7.61 | Val Acc: 76.96%
✅ Best Model Saved: 76.96%
🔥 Epoch 8 | Loss: 6.77 | Val Acc: 77.94%
✅ Best Model Saved: 77.94%
🔥 Epoch 9 | Loss: 6.89 | Val Acc: 78.92%
✅ Best Model Saved: 78.92%
🔥 Epoch 10 | Loss: 5.45 | Val Acc: 78.43%
🔥 Epoch 11 | Loss: 5.55 | Val Acc: 78.92%
🔥 Epoch 12 | Loss: 5.86 | Val Acc: 79.90%
✅ Best Model Saved: 79.90%
🔥 Epoch 13 | Loss: 5.14 | Val Acc: 78.92%
🔥 Epoch 14 | Loss: 4.97 | Val Acc: 79.41%
🔥 Epoch 15 | Loss: 4.34 | Val Acc: 79.90%
🔥 Epoch 16 | Loss: 4.40 | Val Acc: 79.90%
🔥 Epoch 17 | Loss: 4.24 | Val Acc: 80.88%
✅ Best Model Saved: 80.88%
🔥 Epoch 18 | Loss: 3

In [ ]:
def evaluate(model, loader):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, preds = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (preds == labels).sum().item()

    return 100 * correct / total


model.load_state_dict(torch.load("mobilenetv2_best.pth"))

test_acc = evaluate(model, test_loader)

print("🔥 MobileNetV2 Test Accuracy:", test_acc)

🔥 MobileNetV2 Test Accuracy: 92.3076923076923


In [1]:
model.load_state_dict(torch.load("mobilenetv2_best.pth"))

NameError: name 'model' is not defined

In [2]:
from torchvision import models
import torch.nn as nn

model = models.mobilenet_v2(weights=None)

num_features = model.classifier[1].in_features

model.classifier = nn.Sequential(

    nn.Dropout(0.4),

    nn.Linear(num_features, 5)
)

In [3]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

In [4]:
model.load_state_dict(
    torch.load("mobilenetv2_best.pth")
)

print("✅ Model Loaded Successfully")

✅ Model Loaded Successfully


In [5]:
for param in model.features[10:].parameters():
    param.requires_grad = True

In [6]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10
)

In [7]:
model.load_state_dict(torch.load("mobilenetv2_best.pth"))

<All keys matched successfully>

In [8]:
for param in model.features[10:].parameters():
    param.requires_grad = True

In [9]:
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5
)

In [10]:
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10
)

In [11]:
criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

In [12]:
best_acc = 92.31

for epoch in range(10):

    model.train()

    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    # VALIDATION

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, preds = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (preds == labels).sum().item()

    val_acc = 100 * correct / total

    print(f"🔥 FineTune Epoch {epoch+1} | Loss: {running_loss:.2f} | Val Acc: {val_acc:.2f}%")

    # SAVE ONLY IF IMPROVED

    if val_acc > best_acc:

        best_acc = val_acc

        torch.save(
            model.state_dict(),
            "mobilenetv2_finetuned.pth"
        )

        print(f"✅ NEW BEST MODEL SAVED: {best_acc:.2f}%")

    scheduler.step()

NameError: name 'train_loader' is not defined

In [13]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
train_transforms = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(20),

    transforms.RandomAffine(
        degrees=15,
        scale=(0.9,1.1)
    ),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

val_test_transforms = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor(),

    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

In [16]:
train_data = datasets.ImageFolder(
    "/content/drive/MyDrive/dataset/train",
    transform=train_transforms
)

val_data = datasets.ImageFolder(
    "/content/drive/MyDrive/dataset/val",
    transform=val_test_transforms
)

test_data = datasets.ImageFolder(
    "/content/drive/MyDrive/dataset/test",
    transform=val_test_transforms
)

In [17]:
train_loader = DataLoader(
    train_data,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_data,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_data,
    batch_size=32,
    shuffle=False
)

In [18]:
model.load_state_dict(torch.load("mobilenetv2_best.pth"))

<All keys matched successfully>

In [19]:
for param in model.features[10:].parameters():
    param.requires_grad = True

In [20]:
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5
)

In [21]:
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10
)

In [22]:
criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

In [23]:
best_acc = 92.31

for epoch in range(10):

    model.train()

    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    # VALIDATION

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, preds = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (preds == labels).sum().item()

    val_acc = 100 * correct / total

    print(f"🔥 FineTune Epoch {epoch+1} | Loss: {running_loss:.2f} | Val Acc: {val_acc:.2f}%")

    # SAVE ONLY IF IMPROVED

    if val_acc > best_acc:

        best_acc = val_acc

        torch.save(
            model.state_dict(),
            "mobilenetv2_finetuned.pth"
        )

        print(f"✅ NEW BEST MODEL SAVED: {best_acc:.2f}%")

    scheduler.step()

🔥 FineTune Epoch 1 | Loss: 17.02 | Val Acc: 80.39%
🔥 FineTune Epoch 2 | Loss: 16.47 | Val Acc: 80.39%
🔥 FineTune Epoch 3 | Loss: 15.98 | Val Acc: 82.84%
🔥 FineTune Epoch 4 | Loss: 15.85 | Val Acc: 81.37%
🔥 FineTune Epoch 5 | Loss: 15.89 | Val Acc: 82.84%
🔥 FineTune Epoch 6 | Loss: 15.49 | Val Acc: 82.35%
🔥 FineTune Epoch 7 | Loss: 15.65 | Val Acc: 81.86%
🔥 FineTune Epoch 8 | Loss: 15.64 | Val Acc: 82.84%
🔥 FineTune Epoch 9 | Loss: 15.67 | Val Acc: 81.86%
🔥 FineTune Epoch 10 | Loss: 15.54 | Val Acc: 82.84%
